# 01 · Exploratory Data Analysis (H&M)

**Goal:** understand the catalog, the customers and the purchases *before* building anything, so cleaning, splitting and evaluation are based on what the data actually looks like.

**Order:** articles (small) → customers (medium) → transactions (big).

> ⚠️ **License (Kaggle rule 7B):** outputs in this notebook contain real H&M data.
> Before committing: **Clear All Outputs**. Findings go in `docs/eda-report.md`, not in saved outputs.

## 0 · Setup

In [ ]:
from pathlib import Path
import time

import pandas as pd
import matplotlib.pyplot as plt

RAW = Path("../data/raw")              # notebook lives in notebooks/, so go up one folder
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
summary = {}                            # key numbers collected along the way, printed at the end

## 1 · Articles (the catalog)
`article_id` is read as **text** so the leading 0 survives (`0108775015`, which is also the image file name).

In [ ]:
articles = pd.read_csv(RAW / "articles.csv", dtype={"article_id": str})
assert articles["article_id"].str.len().eq(10).all(), "article_id lost its leading 0!"
print("shape (rows, columns):", articles.shape)
summary["n_articles"] = len(articles)
articles.head()

In [ ]:
articles.info(show_counts=True)

### 1.1 · Missing values

In [ ]:
missing = articles.isna().sum()
missing = missing[missing > 0]
print(missing)
print(f"\nproducts with no description: {missing.get('detail_desc', 0)} "
      f"({missing.get('detail_desc', 0) / len(articles):.2%})")
summary["articles_missing_desc"] = int(missing.get("detail_desc", 0))

### 1.2 · Products vs product families
`article_id` = one specific variant (e.g. this top in blue). `product_code` = the family (all colors of that top).

In [ ]:
n_families = articles["product_code"].nunique()
variants = articles.groupby("product_code")["article_id"].count()

print("unique article_id (variants):", articles["article_id"].nunique())
print("unique product_code (families):", n_families)
print(f"variants per family: mean {variants.mean():.2f} | median {variants.median():.0f} | max {variants.max()}")
summary["n_product_families"] = n_families

### 1.3 · What's in the catalog

In [ ]:
for col in ["index_group_name", "product_group_name", "garment_group_name", "colour_group_name"]:
    print(f"\n--- {col}: {articles[col].nunique()} unique values ---")
    print(articles[col].value_counts().head(10))

In [ ]:
articles["product_type_name"].value_counts().head(15).sort_values().plot(
    kind="barh", figsize=(8, 5), title="Top 15 product types in the catalog"
)
plt.xlabel("number of products")
plt.tight_layout()
plt.show()

### 1.4 · How much text does each product have?
Longer descriptions = more for text embeddings to work with.

In [ ]:
desc_words = articles["detail_desc"].fillna("").str.split().str.len()
desc_words.describe()

## 2 · Customers

In [ ]:
customers = pd.read_csv(RAW / "customers.csv")
print("shape:", customers.shape)
summary["n_customers"] = len(customers)
customers.info(show_counts=True)

In [ ]:
print("% missing per column:")
(customers.isna().mean() * 100).round(1)

In [ ]:
for col in ["club_member_status", "fashion_news_frequency"]:
    print(f"\n--- {col} ---")
    print(customers[col].value_counts(dropna=False))

In [ ]:
customers["age"].plot(kind="hist", bins=50, figsize=(8, 4), title="Customer age")
plt.xlabel("age")
plt.tight_layout()
plt.show()

## 3 · Transactions (the big one)
Loading ~3.25 GB efficiently:
- **`dtype`**: tell pandas each column's type up front, so it doesn't guess (guessing is slow and wastes memory). `float32` / `int8` use far less memory than the default 64-bit types.
- **`parse_dates`**: read `t_dat` as real dates, not text.
- We use pandas' **default reader**, not `engine="pyarrow"`. The pyarrow reader is faster, but in testing it **dropped the leading 0 from `article_id` even with `dtype=str`** (it guesses "number" first, then converts). So it's slower but correct. Expect a minute or two.

Then we save a **Parquet copy** in `data/processed/`, so reloading takes seconds instead of a minute or more.

In [ ]:
start_time = time.perf_counter()
transactions = pd.read_csv(
    RAW / "transactions_train.csv",
    dtype={"customer_id": str, "article_id": str, "price": "float32", "sales_channel_id": "int8"},
    parse_dates=["t_dat"],
)
print(f"loaded in {time.perf_counter() - start_time:.1f} s")
print("shape:", transactions.shape)
print(f"memory: {transactions.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
summary["n_transactions"] = len(transactions)

# Safety check: every article_id must be 10 characters (leading 0 kept)
assert transactions["article_id"].str.len().eq(10).all(), "article_id lost its leading 0!"
transactions.head()

In [ ]:
# Cache as Parquet (gitignored). Next time: transactions = pd.read_parquet(PROCESSED / "transactions_full.parquet")
transactions.to_parquet(PROCESSED / "transactions_full.parquet", index=False)
transactions.info()

### 3.1 · Date range (decides the time-based split)

In [ ]:
start, end = transactions["t_dat"].min(), transactions["t_dat"].max()
n_days = (end - start).days + 1
print(f"first purchase: {start.date()} | last purchase: {end.date()} | {n_days} days (~{n_days / 7:.0f} weeks)")
summary["date_range"] = f"{start.date()} to {end.date()}"

In [ ]:
weekly = transactions.groupby(pd.Grouper(key="t_dat", freq="W"))["article_id"].count()
weekly.plot(figsize=(10, 4), title="Purchases per week")
plt.ylabel("purchases")
plt.tight_layout()
plt.show()

### 3.2 · Customers: how much does each one buy?

In [ ]:
per_customer = transactions.groupby("customer_id").size()
summary["n_buying_customers"] = len(per_customer)
print("customers who bought anything:", len(per_customer),
      f"({len(per_customer) / len(customers):.1%} of all customers)")
per_customer.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])

### 3.3 · Articles: which products get bought?

In [ ]:
per_article = transactions.groupby("article_id").size()
never_bought = set(articles["article_id"]) - set(per_article.index)
summary["n_articles_bought"] = len(per_article)
summary["n_articles_never_bought"] = len(never_bought)
print("articles bought at least once:", len(per_article))
print("articles in catalog never bought:", len(never_bought))
per_article.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])

In [ ]:
# Preview of the POPULARITY baseline: the 12 most-bought items, overall vs in the final week
names = articles.set_index("article_id")[["prod_name", "product_type_name", "colour_group_name"]]

last_week_start = end - pd.Timedelta(days=6)
last_week = transactions[transactions["t_dat"] >= last_week_start]

top_overall = per_article.sort_values(ascending=False).head(12).rename("purchases")
top_last_week = last_week.groupby("article_id").size().sort_values(ascending=False).head(12).rename("purchases")

print("Top 12 overall:")
display(names.join(top_overall, how="inner").sort_values("purchases", ascending=False))
print("Top 12 in the final week:")
display(names.join(top_last_week, how="inner").sort_values("purchases", ascending=False))
overlap = len(set(top_overall.index) & set(top_last_week.index))
print(f"overlap between the two lists: {overlap} of 12")

### 3.4 · Data quality checks

In [ ]:
# Do all purchases point to real products and real customers?
print(f"transactions with a known article:  {transactions['article_id'].isin(articles['article_id']).mean():.2%}")
print(f"transactions with a known customer: {transactions['customer_id'].isin(customers['customer_id']).mean():.2%}")

# Exact duplicate rows: in this dataset, buying 2 of the same item on the same day shows up as 2 identical rows
dupes = transactions.duplicated().sum()
summary["exact_duplicate_rows"] = int(dupes)
print(f"exact duplicate rows: {dupes} ({dupes / len(transactions):.2%})")

In [ ]:
print("price (scaled, not real dollars):")
print(transactions["price"].describe())
print("\nsales channel share:")
print(transactions["sales_channel_id"].value_counts(normalize=True).round(3))

### 3.5 · Cold-start preview
How many items sold in the final week had **never** been sold before that week? These are the items "customers also bought" can't recommend.

In [ ]:
first_sale = transactions.groupby("article_id")["t_dat"].min()
new_items = first_sale[first_sale >= last_week_start].index

lw_articles = last_week["article_id"].unique()
share_items = pd.Series(lw_articles).isin(new_items).mean()
share_purchases = last_week["article_id"].isin(new_items).mean()

summary["final_week_new_items"] = len(new_items)
summary["final_week_share_purchases_on_new_items"] = round(float(share_purchases), 4)
print(f"final week: {last_week_start.date()} to {end.date()}")
print(f"items sold for the first time in the final week: {len(new_items)}")
print(f"share of final-week ITEMS that are brand new:     {share_items:.2%}")
print(f"share of final-week PURCHASES on brand-new items: {share_purchases:.2%}")

## 4 · Summary (copy these numbers into `docs/eda-report.md`)

In [ ]:
for key, value in summary.items():
    print(f"{key:45s} {value}")

In [ ]:
# ---- Numbers for docs/eda-report.md (aggregates only, no product rows) ----
print("=== FILES ===")
print("customers rows:", len(customers))
print("transactions rows:", len(transactions))

print("\n=== ARTICLES ===")
print("product families:", articles["product_code"].nunique())
print(f"variants per family: mean {variants.mean():.2f}, median {variants.median():.0f}, max {variants.max()}")
print(f"description words: median {desc_words.median():.0f}")

print("\n=== CUSTOMERS ===")
print(f"buying customers: {len(per_customer)} ({len(per_customer) / len(customers):.1%})")
print("% missing per column:")
print((customers.isna().mean() * 100).round(1).to_string())
print("fashion_news_frequency values:")
print(customers["fashion_news_frequency"].value_counts(dropna=False).to_string())

print("\n=== TRANSACTIONS ===")
print("date range:", start.date(), "to", end.date(), f"({n_days} days, ~{n_days / 7:.0f} weeks)")
print(f"weekly purchases: min {weekly.min()}, median {weekly.median():.0f}, max {weekly.max()} (week of {weekly.idxmax().date()})")
print(f"purchases per customer: median {per_customer.median():.0f}, 90th pct {per_customer.quantile(0.9):.0f}")
print("articles bought at least once:", len(per_article), "| never bought:", len(never_bought))
print(f"exact duplicate rows: {dupes} ({dupes / len(transactions):.2%})")
print(f"known article: {transactions['article_id'].isin(articles['article_id']).mean():.2%} | "
      f"known customer: {transactions['customer_id'].isin(customers['customer_id']).mean():.2%}")

print("\n=== FINDINGS ===")
print("top-12 overlap (overall vs final week):", overlap, "of 12")
print("final week:", last_week_start.date(), "to", end.date())
print("new items in final week:", len(new_items))
print(f"share of final-week purchases on new items: {share_purchases:.2%}")

## Before committing
1. **Restart** the kernel → **Run All**, and check it runs top to bottom without errors.
2. **Clear All Outputs** (license rule 7B), then save.
3. Write findings in plain English in `docs/eda-report.md`.